In [1]:
!pip install gTTS Flask

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 4.7 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


In [2]:
from flask import Flask, request, send_file, render_template_string
from gtts import gTTS
import io
import threading

app = Flask(__name__)

HTML = """
<!doctype html>
<html lang="uk">
<head>
  <meta charset="utf-8">
  <title>Text → Speech</title>
</head>
<body style="font-family: Arial; max-width: 700px; margin: 40px auto;">
  <h2>Лабораторна №4 — Перетворення тексту на мовлення</h2>

  <form method="post" action="/tts">
    <label>Введіть текст:</label><br><br>
    <textarea name="text" rows="6" style="width:100%;" required>{{ default_text }}</textarea><br><br>

    <label>Мова:</label>
    <select name="lang">
      <option value="uk" selected>Українська</option>
      <option value="en">English</option>
      <option value="pl">Polski</option>
      <option value="de">Deutsch</option>
    </select>

    <button type="submit" style="margin-left: 10px;">Озвучити</button>
  </form>

  <p style="color:#555; margin-top:20px;">
    Після натискання буде завантажено MP3-файл.
  </p>
</body>
</html>
"""

@app.route("/", methods=["GET"])
def index():
    return render_template_string(
        HTML,
        default_text="Привіт! Це тест перетворення тексту на мовлення."
    )

@app.route("/tts", methods=["POST"])
def tts():
    text = (request.form.get("text") or "").strip()
    lang = (request.form.get("lang") or "uk").strip()

    if not text:
        return "Помилка: порожній текст", 400

    mp3_bytes = io.BytesIO()
    tts = gTTS(text=text, lang=lang)
    tts.write_to_fp(mp3_bytes)
    mp3_bytes.seek(0)

    return send_file(
        mp3_bytes,
        mimetype="audio/mpeg",
        as_attachment=True,
        download_name="speech.mp3"
    )

def run_app():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

thread = threading.Thread(target=run_app)
thread.start()

In [4]:
!pip install Flask gTTS pyngrok

In [5]:
!ngrok config add-authtoken 3BnrojTZe9flu5lGLwS8ZA9wcaN_2at4cR9ekTSz9uaUKRpDf

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [6]:
from pyngrok import ngrok

public_url = ngrok.connect(5000)
print(public_url)

NgrokTunnel: "https://machinelike-untuneably-louie.ngrok-free.dev" -> "http://localhost:5000"
